# RGCA Baseline Experiment

This notebook is the interactive experiment layer for the first pre-RGCA baseline.

Pipeline:

`Image -> Retriever -> Retrieved Reports -> VLM -> Generated Report`

For now, the repo uses:

- a demo dataset
- a lightweight lexical retriever
- a mock VLM backend that makes retrieval effects visible

That gives us a notebook we can run immediately, then upgrade incrementally with real MIMIC-CXR data and real model backends.

## Notebook Goals

Use this notebook to:

- run the baseline end-to-end
- inspect retrieval examples
- compare `no_retrieval`, `retrieval`, and `mismatch` generations
- confirm that retrieval changes generation behavior
- prepare for replacement with real retrieval and VLM components

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from rgca_baseline.io_utils import read_jsonl
from rgca_baseline.pipeline import load_studies, run_pipeline
from rgca_baseline.retrieval import LexicalRetriever

PROJECT_ROOT

In [ ]:
DATA_PATH = PROJECT_ROOT / 'data' / 'demo' / 'demo_studies.jsonl'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'notebook_demo_run'
TOP_K = 3

DATA_PATH, OUTPUT_DIR

## Load The Dataset

In [ ]:
studies = load_studies(DATA_PATH)
retrieval_pool = [study for study in studies if study.split == 'retrieval_pool']
eval_studies = [study for study in studies if study.split == 'eval']

print('Total studies:', len(studies))
print('Retrieval pool:', len(retrieval_pool))
print('Eval studies:', len(eval_studies))

In [ ]:
for study in studies:
    print({
        'study_id': study.study_id,
        'split': study.split,
        'labels': study.labels,
        'impression': study.impression,
    })

## Inspect Retrieval Behavior

In [ ]:
retriever = LexicalRetriever(retrieval_pool)

for study in eval_studies:
    clean = retriever.retrieve(study, top_k=TOP_K)
    mismatch = retriever.mismatch(study, top_k=TOP_K)
    print('\n=== Target Study:', study.study_id, '===')
    print('Target impression:', study.impression)
    print('Clean retrieval IDs:', clean.retrieved_studies)
    print('Mismatch retrieval IDs:', mismatch.retrieved_studies)

In [ ]:
target = eval_studies[0]
clean = retriever.retrieve(target, top_k=TOP_K)
mismatch = retriever.mismatch(target, top_k=TOP_K)

print('Target:', target.study_id)
print('Target findings:', target.findings)
print('\nClean retrieved reports:')
for idx, report in enumerate(clean.retrieved_reports, start=1):
    print(f'{idx}. {report}')

print('\nMismatch retrieved reports:')
for idx, report in enumerate(mismatch.retrieved_reports, start=1):
    print(f'{idx}. {report}')

## Run The Baseline

In [ ]:
summary = run_pipeline(
    input_path=DATA_PATH,
    output_dir=OUTPUT_DIR,
    mode='all',
    top_k=TOP_K,
)
summary

## Load Generated Outputs

In [ ]:
no_retrieval = read_jsonl(OUTPUT_DIR / 'generations_no_retrieval.jsonl')
retrieval = read_jsonl(OUTPUT_DIR / 'generations_retrieval.jsonl')
mismatch = read_jsonl(OUTPUT_DIR / 'generations_mismatch.jsonl')

print(len(no_retrieval), len(retrieval), len(mismatch))

In [ ]:
by_mode = {
    'no_retrieval': {row['study_id']: row for row in no_retrieval},
    'retrieval': {row['study_id']: row for row in retrieval},
    'mismatch': {row['study_id']: row for row in mismatch},
}

for study in eval_studies:
    print('\n' + '=' * 80)
    print('Study:', study.study_id)
    print('Reference impression:', study.impression)
    print('\nNO RETRIEVAL\n')
    print(by_mode['no_retrieval'][study.study_id]['generated_report'])
    print('\nRETRIEVAL\n')
    print(by_mode['retrieval'][study.study_id]['generated_report'])
    print('\nMISMATCH\n')
    print(by_mode['mismatch'][study.study_id]['generated_report'])

## Inspect Saved Artifacts

In [ ]:
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name)

In [ ]:
summary_path = OUTPUT_DIR / 'run_summary.json'
print(json.loads(summary_path.read_text(encoding='utf-8')))

## How To Upgrade This Notebook

Next replacements for the real experiment:

1. Replace `data/demo/demo_studies.jsonl` with your MIMIC-CXR subset JSONL.
2. Replace `LexicalRetriever` with image-based retrieval using BioMedCLIP plus FAISS.
3. Replace `MockVLMGenerator` with your real VLM backend such as LLaVA-Med.
4. Add a pathology-level hallucination review table from mismatch outputs.

The notebook should stay thin. The reusable implementation should continue to live in `src/rgca_baseline/`.